# 次月スタートMP予測モデル
## 当月日次データから当月末の次月MP（Next_確定+Next_A）を予測

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/goto-a-toast/MP_daily_pred_from_stock-Public/blob/claude/unify-mp-forecast-models-kfJTN/mp_unified_forecast_model.ipynb)

### 概要
当月X日時点の次月データ（Next_確定、Next_A、Next_B、Next_C、Next_D）から、
**当月末時点の Next_確定+Next_A（= 次月スタートMP）** を予測します。

### 予測例
```
当月 2月12日時点
  Next_確定 + Next_A = 4,571万円  ← 今日の実績
          ↓ 予測
  2月末時点の予測値 = 5,200万円   ← 月末までに +629万円 積み上がる
```

### 学習データの考え方
- 過去の「当月X日の Next_確定+A」 → 「同月末日の Next_確定+A」のペアで学習
- 月内のどの時点でも予測可能（月初ほど不確実、月末ほど精度UP）

## 1. ライブラリのインポート

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import calendar
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded!')

## 2. データの読み込み

In [ ]:
try:
    from google.colab import files
    print('Upload CSV file:')
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
except:
    filename = 'progress_data.csv'
    print(f'Using: {filename}')

In [ ]:
try:
    df = pd.read_csv(filename, encoding='utf-8-sig')
except:
    df = pd.read_csv(filename, encoding='shift_jis')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 3. データ前処理

In [ ]:
original_columns = df.columns.tolist()
print(f'Original columns: {original_columns}')

df['date'] = pd.to_datetime(df.iloc[:, 0], errors='coerce')
df = df.dropna(subset=['date']).copy()
df['year_month'] = df['date'].dt.to_period('M')
df['month'] = df['date'].dt.month
df['day']   = df['date'].dt.day

if len(original_columns) >= 13:
    col_mapping = {
        original_columns[1]:  'Current_MP',
        original_columns[7]:  'Next_MP',
        original_columns[8]:  'Next_Kakutei',
        original_columns[9]:  'Next_A',
        original_columns[10]: 'Next_B',
        original_columns[11]: 'Next_C',
        original_columns[12]: 'Next_D'
    }
    print('13-column format')
elif len(original_columns) >= 12:
    col_mapping = {
        original_columns[1]:  'Current_MP',
        original_columns[7]:  'Next_Kakutei',
        original_columns[8]:  'Next_A',
        original_columns[9]:  'Next_B',
        original_columns[10]: 'Next_C',
        original_columns[11]: 'Next_D'
    }
    print('12-column format')
else:
    col_mapping = {
        original_columns[1]: 'Current_MP',
        original_columns[2]: 'Next_Kakutei',
        original_columns[3]: 'Next_A',
        original_columns[4]: 'Next_B',
        original_columns[5]: 'Next_C',
        original_columns[6]: 'Next_D'
    }
    print('7-column format')

df = df.rename(columns=col_mapping)

for col in ['Current_MP', 'Next_MP', 'Next_Kakutei', 'Next_A', 'Next_B', 'Next_C', 'Next_D']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 月末フラグ（各月の最終データ日）
month_end_dates = df.groupby('year_month')['date'].max()
df['is_month_end'] = df.apply(lambda x: x['date'] == month_end_dates[x['year_month']], axis=1)

# KA = Next_確定 + Next_A
df['KA'] = df['Next_Kakutei'] + df['Next_A']

print(f'Daily records : {len(df)}')
print(f'Months        : {df["year_month"].nunique()}')
df[['date', 'year_month', 'KA', 'Next_B', 'Next_C', 'Next_D', 'is_month_end']].head()

## 4. 学習データの作成
各月の月末KA値を取得し、月内の各日のレコードに紐付けます。
完全月（最終データ日 = カレンダー上の月末日）のみを学習に使用します。

In [ ]:
# 各月の月末KA値を取得
ka_end_dict = (
    df[df['is_month_end']]
    .groupby('year_month')['KA']
    .last()
    .to_dict()
)
df['KA_end'] = df['year_month'].map(ka_end_dict)

# 完全月の特定（最終データ日 = 実際のカレンダー月末日）
me_df = df[df['is_month_end']].copy()
me_df['is_actual_end'] = me_df['date'].apply(
    lambda d: d.day == calendar.monthrange(d.year, d.month)[1]
)
complete_months = set(me_df[me_df['is_actual_end']]['year_month'])

# 学習データ：月末以外 + 完全月 + ターゲットあり
train_data = df[
    ~df['is_month_end'] &
    df['KA_end'].notna() &
    df['year_month'].isin(complete_months)
].copy()

print(f'Training data : {len(train_data)} records')
print(f'Months used   : {train_data["year_month"].nunique()} complete months')
print(f'  ({len(complete_months)} / {df["year_month"].nunique()} total months are complete)')
train_data[['date', 'KA', 'KA_end', 'Next_B', 'Next_C', 'Next_D']].head()

## 5. モデル構築・学習
当月X日の次月データから、当月末の KA（Next_確定+Next_A）を予測します。
- 特徴量：現在のKA値、Next_B、Next_C、Next_D、日付（何日目か）、月

In [ ]:
class NextMonthStartMPPredictor:
    """当月X日 -> 当月末の Next_確定+Next_A（次月スタートMP）を予測"""

    def __init__(self):
        self.model = None
        self.monthly_avg = {}

    def train(self, data):
        X = data[['KA', 'Next_B', 'Next_C', 'Next_D', 'day', 'month']]
        y = data['KA_end']

        self.model = RandomForestRegressor(
            n_estimators=150, max_depth=10,
            min_samples_split=5, min_samples_leaf=2, random_state=42
        )
        self.model.fit(X, y)

        # 月別平均（フォールバック用）
        for m in range(1, 13):
            sub = data[data['month'] == m]
            if len(sub) > 0:
                self.monthly_avg[m] = sub['KA_end'].mean()

        pred = self.model.predict(X)
        mae = mean_absolute_error(y, pred)
        r2  = r2_score(y, pred)

        # 特徴量重要度
        feat_names = ['KA', 'Next_B', 'Next_C', 'Next_D', 'day', 'month']
        importance = sorted(
            zip(feat_names, self.model.feature_importances_),
            key=lambda x: x[1], reverse=True
        )
        return {'mae': mae, 'r2': r2, 'importance': importance}

    def predict(self, ka, b, c, d, day, month):
        forecast = float(self.model.predict([[ka, b, c, d, day, month]])[0])
        return {
            'current':  ka,
            'forecast': forecast,
            'growth':   forecast - ka,
            'month':    month,
            'day':      day
        }


predictor = NextMonthStartMPPredictor()
result = predictor.train(train_data)

print('Model Trained!')
print('='*50)
mae_val = result['mae']
r2_val  = result['r2']
print(f'  MAE : {mae_val/10000:>8,.0f} (10K JPY)')
print(f'  R2  : {r2_val:>8.3f}')
print()
print('Feature Importance:')
for name, imp in result['importance']:
    print(f'  {name:10s}: {imp:.1%}')

## 6. 予測
最新日のデータで当月末の次月スタートMPを予測します。

In [ ]:
mn = ['','Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

non_me = df[~df['is_month_end']]
if len(non_me) == 0:
    print('ERROR: No non-month-end data available')
else:
    latest = non_me.iloc[-1]
    month  = int(latest['month'])
    day    = int(latest['day'])
    next_month = (month % 12) + 1

    p = predictor.predict(
        ka=float(latest['KA']),
        b=float(latest['Next_B']),
        c=float(latest['Next_C']),
        d=float(latest['Next_D']),
        day=day, month=month
    )

    current_val  = p['current']
    forecast_val = p['forecast']
    growth_val   = p['growth']

    print('='*60)
    print(f'Next Month Start MP Forecast')
    print(f'  As of: {mn[month]} Day {day}')
    print('='*60)
    print(f'  Current  (today):     {current_val/10000:>10,.0f}  (10K JPY)')
    print(f'  Forecast (month-end): {forecast_val/10000:>10,.0f}  (10K JPY)')
    print(f'  Expected growth:      {growth_val/10000:>+10,.0f}  (10K JPY)')
    print('='*60)
    print(f'  -> By end of {mn[month]}, Next_確定+A is forecast to reach')
    print(f'     {forecast_val/10000:,.0f} (10K JPY)  [= {mn[next_month]} start MP]')

    # 実績（当月末KAが既知なら比較）
    actual_end = latest.get('KA_end', None)
    if pd.notna(actual_end):
        err = abs(forecast_val - actual_end)
        print(f'  Actual month-end: {actual_end/10000:>10,.0f}  (10K JPY)')
        print(f'  Error:            {err/10000:>10,.0f}  (10K JPY)')

## 7. 日次推移グラフ
当月の各日における「今日の実績 vs 月末予測値」の推移を可視化します。

In [ ]:
latest_ym = df['year_month'].max()
latest_month_data = df[(df['year_month'] == latest_ym) & ~df['is_month_end']]

if len(latest_month_data) == 0:
    print('No data available for chart')
else:
    month_val  = int(latest_month_data.iloc[0]['month'])
    next_month = (month_val % 12) + 1

    results = []
    for _, row in latest_month_data.iterrows():
        p = predictor.predict(
            ka=float(row['KA']),
            b=float(row['Next_B']),
            c=float(row['Next_C']),
            d=float(row['Next_D']),
            day=int(row['day']), month=month_val
        )
        results.append({
            'day':      int(row['day']),
            'current':  float(row['KA']),
            'forecast': p['forecast'],
            'growth':   p['growth'],
            'actual':   row.get('KA_end', None)
        })

    sim_df = pd.DataFrame(results)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: current vs forecast
    axes[0].plot(
        sim_df['day'], sim_df['current'] / 10000,
        marker='o', linewidth=2, color='lightblue',
        label='Current Next_確定+A (actual)', alpha=0.9
    )
    axes[0].plot(
        sim_df['day'], sim_df['forecast'] / 10000,
        marker='s', linewidth=2, color='steelblue',
        label=f'Predicted {mn[month_val]}-End value'
    )
    axes[0].fill_between(
        sim_df['day'],
        sim_df['current']  / 10000,
        sim_df['forecast'] / 10000,
        alpha=0.2, color='steelblue', label='Expected growth'
    )
    if pd.notna(sim_df['actual'].iloc[0]):
        actual_val = sim_df['actual'].iloc[0]
        axes[0].axhline(
            y=actual_val / 10000,
            color='red', linestyle='--', linewidth=2,
            label=f'Actual {mn[month_val]}-End value'
        )
    axes[0].set_title(f'Next Month Start MP: {latest_ym} Daily View')
    axes[0].set_xlabel(f'Day of {latest_ym}')
    axes[0].set_ylabel('Next_確定+A (10K JPY)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Right: expected growth by day
    axes[1].bar(
        sim_df['day'], sim_df['growth'] / 10000,
        color='steelblue', alpha=0.7, label='Expected growth to month-end'
    )
    axes[1].axhline(y=0, color='gray', linewidth=0.8)
    axes[1].set_title(f'Expected Growth to {mn[month_val]}-End by Day')
    axes[1].set_xlabel(f'Day of {latest_ym}')
    axes[1].set_ylabel('Expected Growth (10K JPY)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

    # Summary table
    print(f'Daily Summary: {mn[month_val]} -> {mn[next_month]} Start MP')
    print('-'*55)
    print(f'  {"Day":>4}  {"Current":>12}  {"Forecast":>12}  {"Growth":>12}')
    print(f'  {"":>4}  {"(10K JPY)":>12}  {"(10K JPY)":>12}  {"(10K JPY)":>12}')
    print('-'*55)
    for _, row in sim_df.iterrows():
        d    = int(row['day'])
        cur  = int(row['current']  / 10000)
        fore = int(row['forecast'] / 10000)
        grw  = int(row['growth']   / 10000)
        print(f'  {d:>4}  {cur:>12,}  {fore:>12,}  {grw:>+12,}')